In [2]:
import pandas as pd
import numpy as np

In [55]:
train_df = pd.read_csv("../data/processed/train_data.csv")

train_df.shape

(70000, 26)

In [56]:
target = "Bad"

numeric_cols = train_df.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

numeric_cols.remove(target)

categorical_cols = train_df.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical variables:")
print(numeric_cols)

print("\nCategorical variables:")
print(categorical_cols)

Numerical variables:
['Age', 'Monthly_Income', 'Employment_Length_Years', 'Years_at_Address', 'Number_of_Dependents', 'Credit_Utilization', 'Number_of_Credit_Lines', 'Number_of_Open_Loans', 'Number_of_Delinquencies_30', 'Number_of_Delinquencies_60', 'Number_of_Delinquencies_90', 'Total_Debt', 'Debt_to_Income', 'Credit_History_Years', 'Recent_Inquiries_6M', 'Loan_Amount', 'Loan_Tenure_Months', 'Interest_Rate', 'Income_Missing', 'DTI_Missing', 'Employment_Length_Missing']

Categorical variables:
['Employment_Type', 'Home_Ownership', 'Loan_Purpose', 'State']


C:\Users\HP\AppData\Local\Temp\ipykernel_6480\2583648139.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = train_df.select_dtypes(


In [5]:
train_df["Credit_Utilization"].describe()

count    70000.000000
mean         0.463540
std          0.198612
min          0.073000
25%          0.311000
50%          0.459000
75%          0.611000
max          0.894000
Name: Credit_Utilization, dtype: float64

In [6]:
train_df.groupby("Bad")["Credit_Utilization"].describe()

,count,mean,std,min,25%,50%,75%,max
Bad,,,,,,,,
0,65224.0,0.461338,0.197668,0.073,0.309,0.456,0.60800,0.894
1,4776.0,0.493616,0.208793,0.073,0.333,0.494,0.65725,0.894


In [7]:
train_df["Utilization_Bin"] = pd.cut(
    train_df["Credit_Utilization"],
    bins=[-np.inf, 0.20, 0.40, 0.60, 0.80, np.inf],
    labels=[
        "0-20%",
        "20-40%",
        "40-60%",
        "60-80%",
        "80%+"
    ]
)

In [8]:
train_df["Utilization_Bin"].value_counts().sort_index()

Utilization_Bin
0-20%      7129
20-40%    20755
40-60%    23542
60-80%    15071
80%+       3503
Name: count, dtype: int64

In [9]:
bin_analysis = train_df.groupby(
    "Utilization_Bin",
    observed=False
)["Bad"].agg(
    ["count", "sum", "mean"]
)

bin_analysis

,count,sum,mean
Utilization_Bin,,,
0-20%,7129,446,0.062561
20-40%,20755,1237,0.059600
40-60%,23542,1497,0.063588
60-80%,15071,1232,0.081746
80%+,3503,364,0.103911


In [10]:
train_df.groupby("Utilization_Bin")["Bad"].mean()

Utilization_Bin
0-20%     0.062561
20-40%    0.059600
40-60%    0.063588
60-80%    0.081746
80%+      0.103911
Name: Bad, dtype: float64

In [11]:
train_df["DTI_Bin"] = pd.cut(
    train_df["Debt_to_Income"],
    bins=[-np.inf, 0.20, 0.30, 0.40, 0.50, np.inf],
    labels=[
        "0-20%",
        "20-30%",
        "30-40%",
        "40-50%",
        "50%+"
    ]
)

In [12]:
train_df.groupby(
    "DTI_Bin",
    observed=False
)["Bad"].agg(["count", "sum", "mean"])

,count,sum,mean
DTI_Bin,,,
0-20%,2229,140,0.062808
20-30%,11928,716,0.060027
30-40%,25562,1603,0.062710
40-50%,21440,1490,0.069496
50%+,8841,827,0.093541


In [13]:
def calculate_woe_iv(df, feature, target="Bad"):
    
    temp = df[[feature, target]].copy()

    # Total good and bad
    total_good = (temp[target] == 0).sum()
    total_bad = (temp[target] == 1).sum()

    # Good and bad count by bin
    grouped = temp.groupby(
        feature,
        observed=False
    )[target].agg(
        total="count",
        bad="sum"
    )

    grouped["good"] = grouped["total"] - grouped["bad"]

    # Distribution of good and bad
    grouped["dist_good"] = grouped["good"] / total_good
    grouped["dist_bad"] = grouped["bad"] / total_bad

    # Avoid division by zero
    grouped["dist_good"] = grouped["dist_good"].replace(0, 0.000001)
    grouped["dist_bad"] = grouped["dist_bad"].replace(0, 0.000001)

    # WOE
    grouped["WOE"] = np.log(
        grouped["dist_good"] / grouped["dist_bad"]
    )

    # IV contribution
    grouped["IV"] = (
        grouped["dist_good"] - grouped["dist_bad"]
    ) * grouped["WOE"]

    return grouped

In [14]:
utilization_woe = calculate_woe_iv(
    train_df,
    "Utilization_Bin"
)

utilization_woe

,total,bad,good,dist_good,dist_bad,WOE,IV
Utilization_Bin,,,,,,,
0-20%,7129,446,6683,0.102462,0.093384,0.092779,0.000842
20-40%,20755,1237,19518,0.299246,0.259003,0.144424,0.005812
40-60%,23542,1497,22045,0.337989,0.313442,0.075399,0.001851
60-80%,15071,1232,13839,0.212176,0.257956,-0.195372,0.008944
80%+,3503,364,3139,0.048126,0.076214,-0.459718,0.012913


In [15]:
utilization_iv = utilization_woe["IV"].sum()

print("Credit Utilization IV:", utilization_iv)

Credit Utilization IV: 0.030361749349458088


In [16]:
dti_woe = calculate_woe_iv(
    train_df,
    "DTI_Bin"
)

dti_woe

,total,bad,good,dist_good,dist_bad,WOE,IV
DTI_Bin,,,,,,,
0-20%,2229,140,2089,0.032028,0.029313,0.088574,0.000240
20-30%,11928,716,11212,0.171900,0.149916,0.136836,0.003008
30-40%,25562,1603,23959,0.367334,0.335637,0.090243,0.002860
40-50%,21440,1490,19950,0.305869,0.311977,-0.019771,0.000121
50%+,8841,827,8014,0.122869,0.173157,-0.343084,0.017253


In [17]:
dti_iv = dti_woe["IV"].sum()

print("DTI IV:", dti_iv)

DTI IV: 0.02348303532135602


In [18]:
utilization_woe[["dist_good", "dist_bad", "WOE", "IV"]]

,dist_good,dist_bad,WOE,IV
Utilization_Bin,,,,
0-20%,0.102462,0.093384,0.092779,0.000842
20-40%,0.299246,0.259003,0.144424,0.005812
40-60%,0.337989,0.313442,0.075399,0.001851
60-80%,0.212176,0.257956,-0.195372,0.008944
80%+,0.048126,0.076214,-0.459718,0.012913


In [19]:
train_df['Utilization_Bin']

0        20-40%
1        20-40%
2        20-40%
3        40-60%
4        40-60%
          ...  
69995    20-40%
69996    40-60%
69997    40-60%
69998     0-20%
69999    40-60%
Name: Utilization_Bin, Length: 70000, dtype: category
Categories (5, str): ['0-20%' < '20-40%' < '40-60%' < '60-80%' < '80%+']

In [20]:
train_df["Credit_Utilization"]

0        0.350
1        0.284
2        0.382
3        0.574
4        0.546
         ...  
69995    0.278
69996    0.428
69997    0.453
69998    0.083
69999    0.513
Name: Credit_Utilization, Length: 70000, dtype: float64

In [21]:
features_to_check = [
    "Utilization_Bin",
    "DTI_Bin"
]

iv_results = []

for feature in features_to_check:

    woe_table = calculate_woe_iv(
        train_df,
        feature
    )

    iv = woe_table["IV"].sum()

    iv_results.append({
        "Feature": feature,
        "IV": iv
    })

iv_df = pd.DataFrame(iv_results)

iv_df = iv_df.sort_values(
    "IV",
    ascending=False
)

iv_df

,Feature,IV
0,Utilization_Bin,0.030362
1,DTI_Bin,0.023483


In [22]:
# STEP 8: Automatic Binning + WOE + IV

In [23]:
def create_numeric_bins(df, feature, n_bins=10):
    
    temp = df[[feature, "Bad"]].copy()

    # Remove missing values temporarily
    temp = temp.dropna(subset=[feature])

    # Create quantile bins
    temp[f"{feature}_Bin"] = pd.qcut(
        temp[feature],
        q=n_bins,
        duplicates="drop"
    )

    return temp

In [24]:
utilization_bins = create_numeric_bins(
    train_df,
    "Credit_Utilization",
    n_bins=10
)

utilization_bins.head()

,Credit_Utilization,Bad,Credit_Utilization_Bin
0,0.350,0,"(0.343, 0.402]"
1,0.284,0,"(0.277, 0.343]"
2,0.382,0,"(0.343, 0.402]"
3,0.574,0,"(0.517, 0.577]"
4,0.546,0,"(0.517, 0.577]"


In [25]:
utilization_bins.groupby(
    "Credit_Utilization_Bin",
    observed=False
)["Bad"].agg(
    ["count", "sum", "mean"]
)

,count,sum,mean
Credit_Utilization_Bin,,,
"(0.072, 0.199]",7049,438,0.062136
"(0.199, 0.277]",6995,413,0.059042
"(0.277, 0.343]",7062,429,0.060748
"(0.343, 0.402]",6996,413,0.059034
"(0.402, 0.459]",7014,418,0.059595
"(0.459, 0.517]",7017,454,0.064700
"(0.517, 0.577]",6914,475,0.068701
"(0.577, 0.648]",6997,484,0.069173
"(0.648, 0.736]",6986,556,0.079588


In [62]:
def calculate_numeric_woe_iv(df, feature, target="Bad", n_bins=10):

    temp = df[[feature, target]].copy()

    # Handle missing values separately
    temp["Bin"] = temp[feature].astype(object)
    temp.loc[temp[feature].isna(), "Bin"] = "Missing"

    # Number of unique non-missing values
    n_unique = temp[feature].nunique()

    # Low-cardinality variable
    if n_unique <= 5:

        temp.loc[
            temp[feature].notna(),
            "Bin"
        ] = temp.loc[
            temp[feature].notna(),
            feature
        ].astype(str)

    # Continuous variable
    else:

        non_missing = temp[feature].notna()

        temp.loc[non_missing, "Bin"] = pd.qcut(
            temp.loc[non_missing, feature],
            q=n_bins,
            duplicates="drop"
        )

    total_good = (temp[target] == 0).sum()
    total_bad = (temp[target] == 1).sum()

    grouped = (
        temp.groupby(
            "Bin",
            dropna=False,
            observed=False
        )[target]
        .agg(
            total="count",
            bad="sum"
        )
    )

    grouped["good"] = grouped["total"] - grouped["bad"]

    grouped["dist_good"] = (
        grouped["good"] / total_good
    )

    grouped["dist_bad"] = (
        grouped["bad"] / total_bad
    )

    grouped["dist_good"] = grouped["dist_good"].replace(
        0, 0.000001
    )

    grouped["dist_bad"] = grouped["dist_bad"].replace(
        0, 0.000001
    )

    grouped["WOE"] = np.log(
        grouped["dist_good"] /
        grouped["dist_bad"]
    )

    grouped["IV"] = (
        grouped["dist_good"] -
        grouped["dist_bad"]
    ) * grouped["WOE"]

    return grouped

In [64]:
result = calculate_numeric_woe_iv(
    train_df,
    "Number_of_Delinquencies_30"
)

display(result)

print("Total IV:", result["IV"].sum())

,total,bad,good,dist_good,dist_bad,WOE,IV
Bin,,,,,,,
0.0,65967,4403,61564,0.943886,0.921901,0.023567,0.000518
1.0,2373,194,2179,0.033408,0.040620,-0.195461,0.001410
2.0,906,91,815,0.012495,0.019054,-0.421896,0.002767
3.0,384,47,337,0.005167,0.009841,-0.644289,0.003011
4.0,171,26,145,0.002223,0.005444,-0.895587,0.002884
Missing,199,15,184,0.002821,0.003141,-0.107339,0.000034


Total IV: 0.010624857406817345


In [65]:
result = calculate_numeric_woe_iv(
    train_df,
    "Number_of_Delinquencies_30"
)

display(result)

,total,bad,good,dist_good,dist_bad,WOE,IV
Bin,,,,,,,
0.0,65967,4403,61564,0.943886,0.921901,0.023567,0.000518
1.0,2373,194,2179,0.033408,0.040620,-0.195461,0.001410
2.0,906,91,815,0.012495,0.019054,-0.421896,0.002767
3.0,384,47,337,0.005167,0.009841,-0.644289,0.003011
4.0,171,26,145,0.002223,0.005444,-0.895587,0.002884
Missing,199,15,184,0.002821,0.003141,-0.107339,0.000034


In [66]:
result_60 = calculate_numeric_woe_iv(
    train_df,
    "Number_of_Delinquencies_60"
)

display(result_60)

print("Total IV:", result_60["IV"].sum())

,total,bad,good,dist_good,dist_bad,WOE,IV
Bin,,,,,,,
0,68679,4604,64075,0.982384,0.963987,0.018905,0.000348
1,916,95,821,0.012587,0.019891,-0.457578,0.003342
2,295,52,243,0.003726,0.010888,-1.072406,0.007681
3,110,25,85,0.001303,0.005235,-1.390449,0.005466


Total IV: 0.016836831005790424


In [67]:
result_90 = calculate_numeric_woe_iv(
    train_df,
    "Number_of_Delinquencies_90"
)

display(result_90)

print("Total IV:", result_90["IV"].sum())

,total,bad,good,dist_good,dist_bad,WOE,IV
Bin,,,,,,,
0,69606,4704,64902,0.995063,0.984925,0.010241,0.000104
1,328,48,280,0.004293,0.010050,-0.850636,0.004897
2,66,24,42,0.000644,0.005025,-2.054608,0.009002


Total IV: 0.01400287017562162


In [68]:
delinquency_cols = [
    "Number_of_Delinquencies_30",
    "Number_of_Delinquencies_60",
    "Number_of_Delinquencies_90"
]

train_df[delinquency_cols].corr()

,Number_of_Delinquencies_30,Number_of_Delinquencies_60,Number_of_Delinquencies_90
Number_of_Delinquencies_30,1.000000,0.061423,0.000953
Number_of_Delinquencies_60,0.061423,1.000000,0.051536
Number_of_Delinquencies_90,0.000953,0.051536,1.000000


In [70]:
candidate_features = [
    "Credit_History_Years",
    "Age",
    "Credit_Utilization",
    "Interest_Rate",
    "Debt_to_Income",
    "Employment_Length_Years",
    "Monthly_Income",
    "Number_of_Delinquencies_30",
    "Number_of_Delinquencies_60",
    "Number_of_Delinquencies_90"
]

corr_matrix = train_df[candidate_features].corr()

corr_matrix

,Credit_History_Years,Age,Credit_Utilization,Interest_Rate,Debt_to_Income,Employment_Length_Years,Monthly_Income,Number_of_Delinquencies_30,Number_of_Delinquencies_60,Number_of_Delinquencies_90
Credit_History_Years,1.000000,0.968070,0.003988,0.000144,0.001187,-0.003960,0.402301,-0.015251,-0.018991,-0.010146
Age,0.968070,1.000000,0.002384,-0.001080,0.001184,-0.005176,0.414775,-0.015193,-0.018528,-0.009151
Credit_Utilization,0.003988,0.002384,1.000000,0.805863,0.442361,-0.002405,-0.002930,0.012949,0.009964,0.007642
Interest_Rate,0.000144,-0.001080,0.805863,1.000000,0.488363,-0.003319,-0.004254,0.012230,0.013911,0.006586
Debt_to_Income,0.001187,0.001184,0.442361,0.488363,1.000000,-0.007940,-0.005917,0.010176,0.014056,0.002986
Employment_Length_Years,-0.003960,-0.005176,-0.002405,-0.003319,-0.007940,1.000000,0.240885,-0.008122,-0.012510,-0.009054
Monthly_Income,0.402301,0.414775,-0.002930,-0.004254,-0.005917,0.240885,1.000000,-0.009421,-0.013010,-0.000630
Number_of_Delinquencies_30,-0.015251,-0.015193,0.012949,0.012230,0.010176,-0.008122,-0.009421,1.000000,0.061423,0.000953
Number_of_Delinquencies_60,-0.018991,-0.018528,0.009964,0.013911,0.014056,-0.012510,-0.013010,0.061423,1.000000,0.051536
Number_of_Delinquencies_90,-0.010146,-0.009151,0.007642,0.006586,0.002986,-0.009054,-0.000630,0.000953,0.051536,1.000000


In [72]:
final_features = [
    "Credit_History_Years",
    "Credit_Utilization",
    "Debt_to_Income",
    "Employment_Length_Years",
    "Monthly_Income",
    "Number_of_Delinquencies_30",
    "Number_of_Delinquencies_60",
    "Number_of_Delinquencies_90"
]

print("Final candidate features:")
for feature in final_features:
    print("-", feature)

Final candidate features:
- Credit_History_Years
- Credit_Utilization
- Debt_to_Income
- Employment_Length_Years
- Monthly_Income
- Number_of_Delinquencies_30
- Number_of_Delinquencies_60
- Number_of_Delinquencies_90


In [73]:
def fit_woe_transformer(df, feature, target="Bad", n_bins=10):

    temp = df[[feature, target]].copy()

    # Missing values
    temp["Bin"] = temp[feature].astype(object)
    temp.loc[temp[feature].isna(), "Bin"] = "Missing"

    # Number of unique non-missing values
    n_unique = temp[feature].nunique()

    # Low-cardinality variable
    if n_unique <= 5:

        temp.loc[temp[feature].notna(), "Bin"] = (
            temp.loc[temp[feature].notna(), feature].astype(str)
        )

        bin_edges = None

    # Continuous variable
    else:

        non_missing = temp[feature].notna()

        temp.loc[non_missing, "Bin"], bin_edges = pd.qcut(
            temp.loc[non_missing, feature],
            q=n_bins,
            retbins=True,
            duplicates="drop"
        )

    total_good = (temp[target] == 0).sum()
    total_bad = (temp[target] == 1).sum()

    grouped = (
        temp.groupby(
            "Bin",
            dropna=False,
            observed=False
        )[target]
        .agg(
            total="count",
            bad="sum"
        )
    )

    grouped["good"] = grouped["total"] - grouped["bad"]

    grouped["dist_good"] = grouped["good"] / total_good
    grouped["dist_bad"] = grouped["bad"] / total_bad

    grouped["dist_good"] = grouped["dist_good"].replace(
        0, 0.000001
    )

    grouped["dist_bad"] = grouped["dist_bad"].replace(
        0, 0.000001
    )

    grouped["WOE"] = np.log(
        grouped["dist_good"] / grouped["dist_bad"]
    )

    grouped["IV"] = (
        grouped["dist_good"] -
        grouped["dist_bad"]
    ) * grouped["WOE"]

    return {
        "bin_edges": bin_edges,
        "woe_table": grouped
    }

In [119]:
woe_transformers = {}

for feature in final_features:

    woe_transformers[feature] = fit_woe_transformer(
        train_df,
        feature
    )

print("WOE transformers created successfully.")

WOE transformers created successfully.


In [75]:
woe_transformers["Credit_Utilization"]["woe_table"]

,total,bad,good,dist_good,dist_bad,WOE,IV
Bin,,,,,,,
"(0.072, 0.199]",7049,438,6611,0.101358,0.091709,0.100047,0.000965
"(0.199, 0.277]",6995,413,6582,0.100914,0.086474,0.154422,0.002230
"(0.277, 0.343]",7062,429,6633,0.101696,0.089824,0.124131,0.001474
"(0.343, 0.402]",6996,413,6583,0.100929,0.086474,0.154574,0.002234
"(0.402, 0.459]",7014,418,6596,0.101128,0.087521,0.144513,0.001966
"(0.459, 0.517]",7017,454,6563,0.100622,0.095059,0.056882,0.000316
"(0.517, 0.577]",6914,475,6439,0.098721,0.099456,-0.007410,0.000005
"(0.577, 0.648]",6997,484,6513,0.099856,0.101340,-0.014754,0.000022
"(0.648, 0.736]",6986,556,6430,0.098583,0.116415,-0.166263,0.002965


In [76]:
woe_transformers["Number_of_Delinquencies_30"]["woe_table"]

,total,bad,good,dist_good,dist_bad,WOE,IV
Bin,,,,,,,
0.0,65967,4403,61564,0.943886,0.921901,0.023567,0.000518
1.0,2373,194,2179,0.033408,0.040620,-0.195461,0.001410
2.0,906,91,815,0.012495,0.019054,-0.421896,0.002767
3.0,384,47,337,0.005167,0.009841,-0.644289,0.003011
4.0,171,26,145,0.002223,0.005444,-0.895587,0.002884
Missing,199,15,184,0.002821,0.003141,-0.107339,0.000034


In [118]:
def transform_with_woe(df, feature, transformer):

    result = df.copy()

    bin_edges = transformer["bin_edges"]
    woe_table = transformer["woe_table"]

    if bin_edges is None:

        result["Bin"] = result[feature].astype(object)

        result.loc[result[feature].isna(), "Bin"] = "Missing"

        result.loc[result[feature].notna(), "Bin"] = (
            result.loc[result[feature].notna(), feature].astype(str)
        )

        woe_dict = woe_table["WOE"].to_dict()
        result[f"{feature}_WOE"] = result["Bin"].map(woe_dict)

    else:

        # Find bin number instead of matching interval labels
        values = result[feature]

        bin_number = np.digitize(
            values.fillna(bin_edges[0]),
            bin_edges[1:-1],
            right=True
        )

        woe_values = woe_table["WOE"].iloc[
            np.clip(bin_number, 0, len(woe_table) - 1)
        ].values

        result[f"{feature}_WOE"] = woe_values

        # Missing values use the Missing WOE
        if "Missing" in woe_table.index:
            result.loc[
                values.isna(),
                f"{feature}_WOE"
            ] = woe_table.loc["Missing", "WOE"]

    return result

In [89]:
test_30 = transform_with_woe(
    train_df,
    "Number_of_Delinquencies_30",
    woe_transformers["Number_of_Delinquencies_30"]
)

In [90]:
test_30[
    [
        "Number_of_Delinquencies_30",
        "Number_of_Delinquencies_30_WOE",
        "Bad"
    ]
].head(10)

,Number_of_Delinquencies_30,Number_of_Delinquencies_30_WOE,Bad
0,0.0,0.023567,0
1,0.0,0.023567,0
2,NaN,-0.107339,0
3,0.0,0.023567,0
4,0.0,0.023567,0
5,0.0,0.023567,0
6,0.0,0.023567,0
7,0.0,0.023567,0
8,0.0,0.023567,0
9,0.0,0.023567,0


In [91]:
print(train_df["Number_of_Delinquencies_30"].unique())

[ 0. nan  1.  2.  4.  3.]


In [92]:
print(
    woe_transformers["Number_of_Delinquencies_30"]["woe_table"]["WOE"]
)

Bin
0.0        0.023567
1.0       -0.195461
2.0       -0.421896
3.0       -0.644289
4.0       -0.895587
Missing   -0.107339
Name: WOE, dtype: float64


In [93]:
print(
    woe_transformers["Number_of_Delinquencies_30"]["woe_table"].index
)

Index(['0.0', '1.0', '2.0', '3.0', '4.0', 'Missing'], dtype='str', name='Bin')


In [83]:
test_30[
    [
        "Number_of_Delinquencies_30",
        "Number_of_Delinquencies_30_WOE",
        "Bad"
    ]
].head(10).to_string()

'   Number_of_Delinquencies_30  Number_of_Delinquencies_30_WOE  Bad\n0                         0.0                        0.023567    0\n1                         0.0                        0.023567    0\n2                         NaN                       -0.107339    0\n3                         0.0                        0.023567    0\n4                         0.0                        0.023567    0\n5                         0.0                        0.023567    0\n6                         0.0                        0.023567    0\n7                         0.0                        0.023567    0\n8                         0.0                        0.023567    0\n9                         0.0                        0.023567    0'

In [84]:
print("Raw unique values:")
print(train_df["Number_of_Delinquencies_30"].unique())

Raw unique values:
[ 0. nan  1.  2.  4.  3.]


In [85]:
print("WOE mapping:")
print(
    woe_transformers["Number_of_Delinquencies_30"]["woe_table"][
        ["WOE"]
    ]
)

WOE mapping:
              WOE
Bin              
0.0      0.023567
1.0     -0.195461
2.0     -0.421896
3.0     -0.644289
4.0     -0.895587
Missing -0.107339


In [94]:
train_woe = train_df.copy()

for feature in final_features:
    train_woe = transform_with_woe(
        train_woe,
        feature,
        woe_transformers[feature]
    )

In [95]:
woe_columns = [
    f"{feature}_WOE"
    for feature in final_features
]

train_woe[woe_columns].head()

,Credit_History_Years_WOE,Credit_Utilization_WOE,Debt_to_Income_WOE,Employment_Length_Years_WOE,Monthly_Income_WOE,Number_of_Delinquencies_30_WOE,Number_of_Delinquencies_60_WOE,Number_of_Delinquencies_90_WOE
0,-0.088343,0.154574,-0.116979,-0.066406,0.142236,0.023567,0.018905,0.010241
1,-0.335751,0.124131,0.150427,0.065696,-0.156247,0.023567,0.018905,0.010241
2,0.056292,0.154574,0.038674,-0.066406,0.087442,-0.107339,0.018905,0.010241
3,-0.006497,-0.007410,-0.001783,-0.005268,0.119870,0.023567,0.018905,0.010241
4,0.325807,-0.007410,0.150427,0.269160,0.142236,0.023567,0.018905,0.010241


In [96]:
train_woe[woe_columns].isna().sum()

Credit_History_Years_WOE          0
Credit_Utilization_WOE            0
Debt_to_Income_WOE                0
Employment_Length_Years_WOE       0
Monthly_Income_WOE                0
Number_of_Delinquencies_30_WOE    0
Number_of_Delinquencies_60_WOE    0
Number_of_Delinquencies_90_WOE    0
dtype: int64

In [97]:
train_woe.filter(like="_WOE").head()

,Credit_History_Years_WOE,Credit_Utilization_WOE,Debt_to_Income_WOE,Employment_Length_Years_WOE,Monthly_Income_WOE,Number_of_Delinquencies_30_WOE,Number_of_Delinquencies_60_WOE,Number_of_Delinquencies_90_WOE
0,-0.088343,0.154574,-0.116979,-0.066406,0.142236,0.023567,0.018905,0.010241
1,-0.335751,0.124131,0.150427,0.065696,-0.156247,0.023567,0.018905,0.010241
2,0.056292,0.154574,0.038674,-0.066406,0.087442,-0.107339,0.018905,0.010241
3,-0.006497,-0.007410,-0.001783,-0.005268,0.119870,0.023567,0.018905,0.010241
4,0.325807,-0.007410,0.150427,0.269160,0.142236,0.023567,0.018905,0.010241


In [99]:
test_df = pd.read_csv("../data/processed/test_data.csv")

In [120]:
test_woe = test_df.copy()

for feature in final_features:
    test_woe = transform_with_woe(
        test_woe,
        feature,
        woe_transformers[feature]
    )

test_woe.head()

,Age,Monthly_Income,Employment_Length_Years,Employment_Type,Home_Ownership,Years_at_Address,Number_of_Dependents,Credit_Utilization,Number_of_Credit_Lines,Number_of_Open_Loans,...,Bad,Credit_History_Years_WOE,Credit_Utilization_WOE,Debt_to_Income_WOE,Employment_Length_Years_WOE,Monthly_Income_WOE,Bin,Number_of_Delinquencies_30_WOE,Number_of_Delinquencies_60_WOE,Number_of_Delinquencies_90_WOE
0,48,95847.00,2.5,Contract,Mortgage,8.1,0,0.574,8,1,...,0,0.254713,-0.007410,0.150427,-0.185088,0.181361,0,0.023567,0.018905,0.010241
1,26,77642.00,8.8,Business,Mortgage,2.1,2,0.222,8,0,...,0,-0.335751,0.154422,0.197812,0.065047,-0.171877,0,0.023567,0.018905,0.010241
2,30,59607.00,4.8,Salaried,Own,4.4,0,0.134,4,4,...,0,-0.197876,0.100047,-0.001783,-0.066406,-0.114209,0,0.023567,0.018905,0.010241
3,35,50711.00,10.8,Unemployed,Mortgage,7.9,3,0.358,6,1,...,0,-0.197876,0.154574,0.108215,0.178351,-0.156247,0,0.023567,0.018905,0.010241
4,21,40209.94,2.7,Business,Rent,5.2,0,0.375,12,1,...,0,-0.335751,0.154574,0.002974,-0.185088,-0.156247,0,0.023567,0.018905,0.010241


In [121]:
test_woe.filter(like="_WOE").isna().sum()

Credit_History_Years_WOE          0
Credit_Utilization_WOE            0
Debt_to_Income_WOE                0
Employment_Length_Years_WOE       0
Monthly_Income_WOE                0
Number_of_Delinquencies_30_WOE    0
Number_of_Delinquencies_60_WOE    0
Number_of_Delinquencies_90_WOE    0
dtype: int64

In [123]:
woe_features = [f"{feature}_WOE" for feature in final_features]

train_model = train_woe[woe_features + ["Bad"]].copy()
test_model = test_woe[woe_features + ["Bad"]].copy()

In [124]:
train_model.shape, test_model.shape

((70000, 9), (30000, 9))

In [125]:
train_model.head()

,Credit_History_Years_WOE,Credit_Utilization_WOE,Debt_to_Income_WOE,Employment_Length_Years_WOE,Monthly_Income_WOE,Number_of_Delinquencies_30_WOE,Number_of_Delinquencies_60_WOE,Number_of_Delinquencies_90_WOE,Bad
0,-0.088343,0.154574,-0.116979,-0.066406,0.142236,0.023567,0.018905,0.010241,0
1,-0.335751,0.124131,0.150427,0.065696,-0.156247,0.023567,0.018905,0.010241,0
2,0.056292,0.154574,0.038674,-0.066406,0.087442,-0.107339,0.018905,0.010241,0
3,-0.006497,-0.007410,-0.001783,-0.005268,0.119870,0.023567,0.018905,0.010241,0
4,0.325807,-0.007410,0.150427,0.269160,0.142236,0.023567,0.018905,0.010241,0


In [126]:
train_model.to_csv(
    "../data/processed/train_woe.csv",
    index=False
)

test_model.to_csv(
    "../data/processed/test_woe.csv",
    index=False
)

In [127]:
import os

print(os.path.exists("../data/processed/train_woe.csv"))
print(os.path.exists("../data/processed/test_woe.csv"))

True
True


In [128]:
challenger_features = [
    "Age",
    "Interest_Rate"
]

In [129]:
challenger_woe_transformers = {}

for feature in challenger_features:
    challenger_woe_transformers[feature] = fit_woe_transformer(
        train_df,
        feature
    )

In [130]:
train_challenger = train_df.copy()

for feature in challenger_features:
    train_challenger = transform_with_woe(
        train_challenger,
        feature,
        challenger_woe_transformers[feature]
    )

In [131]:
test_challenger = test_df.copy()

for feature in challenger_features:
    test_challenger = transform_with_woe(
        test_challenger,
        feature,
        challenger_woe_transformers[feature]
    )

In [132]:
train_challenger[
    "Age_WOE"
] = train_challenger["Age_WOE"]

train_challenger[
    "Interest_Rate_WOE"
] = train_challenger["Interest_Rate_WOE"]

In [133]:
train_challenger.to_csv(
    "../data/processed/train_challenger_woe.csv",
    index=False
)

test_challenger.to_csv(
    "../data/processed/test_challenger_woe.csv",
    index=False
)